# 中芯国际 A 股与港股近一年交易数据分析

本 Notebook 复现 `task1` 中数据整理和绘图分析过程。

- A 股代码：`688981.SH`
- 港股代码：`00981.HK`
- 数据区间：`2025-07-02` 至 `2026-07-02`
- 原始数据来源：Tushare MCP 获取后保存为 CSV

为避免暴露 Tushare token，并避免重复触发接口频率限制，本 Notebook 从已经保存好的 CSV 文件开始分析。

运行依赖：`pandas`、`numpy`、`matplotlib`。

## 1. 准备环境

使用 `pandas` 进行数据处理，使用 `matplotlib` 绘图。

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

DATA_DIR = Path('.')
A_CSV = DATA_DIR / 'smic_a_daily.csv'
HK_CSV = DATA_DIR / 'smic_hk_daily.csv'

A_CODE = '688981.SH'
HK_CODE = '00981.HK'

## 2. 读取 Tushare 导出的 CSV

两个 CSV 均包含：交易代码、交易日期、开高低收、昨收、涨跌额、涨跌幅、成交量、成交额。

In [ ]:
def load_daily_csv(path: Path, market: str) -> pd.DataFrame:
    df = pd.read_csv(path, encoding='utf-8-sig')
    df['trade_date'] = pd.to_datetime(df['trade_date'].astype(str), format='%Y%m%d')
    df = df.sort_values('trade_date').reset_index(drop=True)
    df['market'] = market
    numeric_cols = ['open', 'high', 'low', 'close', 'pre_close', 'change', 'pct_chg', 'vol', 'amount']
    df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')
    return df

a_df = load_daily_csv(A_CSV, 'A股')
hk_df = load_daily_csv(HK_CSV, '港股')

print('A股数据量:', len(a_df), '起止日期:', a_df['trade_date'].min().date(), a_df['trade_date'].max().date())
print('港股数据量:', len(hk_df), '起止日期:', hk_df['trade_date'].min().date(), hk_df['trade_date'].max().date())

display(a_df.head())
display(hk_df.head())

## 3. 衍生指标计算

这里补充以下指标：

- `ma5` / `ma20`：5 日、20 日均线
- `vol_ma20`：20 日平均成交量
- `drawdown_pct`：相对历史阶段高点的回撤
- `volatility20`：20 日滚动波动率
- `close_index`：将首日收盘价归一化为 100，用于 A/H 对比

In [ ]:
def add_indicators(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out['ma5'] = out['close'].rolling(5, min_periods=1).mean()
    out['ma20'] = out['close'].rolling(20, min_periods=1).mean()
    out['vol_ma20'] = out['vol'].rolling(20, min_periods=1).mean()
    out['cum_peak'] = out['close'].cummax()
    out['drawdown_pct'] = (out['close'] / out['cum_peak'] - 1) * 100
    out['volatility20'] = out['pct_chg'].rolling(20, min_periods=2).std(ddof=0)
    out['close_index'] = out['close'] / out['close'].iloc[0] * 100
    return out

a_df = add_indicators(a_df)
hk_df = add_indicators(hk_df)
combined = pd.concat([a_df, hk_df], ignore_index=True)

combined[['market', 'ts_code', 'trade_date', 'close', 'ma5', 'ma20', 'drawdown_pct', 'volatility20']].tail()

## 4. 总体统计摘要

先对两个市场分别做基础统计，便于理解后续图表。

In [ ]:
def market_summary(df: pd.DataFrame) -> pd.Series:
    first_close = df['close'].iloc[0]
    last_close = df['close'].iloc[-1]
    return pd.Series({
        '代码': df['ts_code'].iloc[0],
        '交易日数量': len(df),
        '起始日期': df['trade_date'].min().date(),
        '结束日期': df['trade_date'].max().date(),
        '首日收盘价': first_close,
        '最新收盘价': last_close,
        '累计涨跌幅%': (last_close / first_close - 1) * 100,
        '上涨天数': (df['pct_chg'] > 0).sum(),
        '下跌天数': (df['pct_chg'] < 0).sum(),
        '上涨占比%': (df['pct_chg'] > 0).mean() * 100,
        '最大回撤%': df['drawdown_pct'].min(),
        '平均成交量': df['vol'].mean(),
        '平均成交额': df['amount'].mean(),
    })

summary = pd.DataFrame({
    'A股': market_summary(a_df),
    '港股': market_summary(hk_df),
}).T

display(summary)

## 5. 绘图辅助函数

下面定义 K 线图、成交量图、收益率分布等绘图函数。

In [ ]:
UP_COLOR = '#d92d20'
DOWN_COLOR = '#039855'
BLUE = '#2563eb'
ORANGE = '#d97706'
PURPLE = '#7c3aed'

def plot_candlestick(ax, df: pd.DataFrame, title: str):
    x = np.arange(len(df))
    width = 0.62
    for i, row in df.iterrows():
        color = UP_COLOR if row['close'] >= row['open'] else DOWN_COLOR
        ax.vlines(x[i], row['low'], row['high'], color=color, linewidth=1)
        lower = min(row['open'], row['close'])
        height = abs(row['close'] - row['open']) or 0.01
        ax.add_patch(Rectangle((x[i] - width / 2, lower), width, height, facecolor=color, edgecolor=color, alpha=0.9))
    ax.plot(x, df['ma5'], color=BLUE, linewidth=1.5, label='MA5')
    ax.plot(x, df['ma20'], color=ORANGE, linewidth=1.5, label='MA20')
    tick_idx = np.linspace(0, len(df) - 1, 6, dtype=int)
    ax.set_xticks(tick_idx)
    ax.set_xticklabels(df.loc[tick_idx, 'trade_date'].dt.strftime('%Y-%m-%d'), rotation=0)
    ax.set_title(title)
    ax.set_ylabel('价格')
    ax.legend()

def plot_volume(ax, df: pd.DataFrame, title: str):
    x = np.arange(len(df))
    colors = np.where(df['close'] >= df['open'], UP_COLOR, DOWN_COLOR)
    ax.bar(x, df['vol'], color=colors, alpha=0.75, label='成交量')
    ax.plot(x, df['vol_ma20'], color=BLUE, linewidth=1.5, label='20日均量')
    tick_idx = np.linspace(0, len(df) - 1, 6, dtype=int)
    ax.set_xticks(tick_idx)
    ax.set_xticklabels(df.loc[tick_idx, 'trade_date'].dt.strftime('%Y-%m-%d'))
    ax.set_title(title)
    ax.set_ylabel('成交量')
    ax.legend()

def plot_single_market(df: pd.DataFrame, market_name: str):
    fig, axes = plt.subplots(3, 2, figsize=(16, 14))
    axes = axes.ravel()
    plot_candlestick(axes[0], df, f'{market_name} 图1：K线与5/20日均线')
    plot_volume(axes[1], df, f'{market_name} 图2：成交量与20日均量')
    axes[2].plot(df['trade_date'], df['close'], color=BLUE, label='收盘价')
    axes[2].set_title(f'{market_name} 图3：每日收盘价曲线')
    axes[2].set_ylabel('收盘价')
    axes[2].legend()
    axes[3].plot(df['trade_date'], df['drawdown_pct'], color=PURPLE)
    axes[3].fill_between(df['trade_date'], df['drawdown_pct'], 0, color=PURPLE, alpha=0.15)
    axes[3].set_title(f'{market_name} 图4：相对阶段高点回撤')
    axes[3].set_ylabel('回撤%')
    axes[4].hist(df['pct_chg'], bins=24, color='#64748b', edgecolor='white')
    axes[4].axvline(0, color='black', linewidth=1)
    axes[4].set_title(f'{market_name} 图5：日收益率分布')
    axes[4].set_xlabel('日涨跌幅%')
    axes[4].set_ylabel('交易日数量')
    axes[5].plot(df['trade_date'], df['volatility20'], color=ORANGE)
    axes[5].set_title(f'{market_name} 图6：20日滚动波动率')
    axes[5].set_ylabel('波动率%')
    fig.tight_layout()
    return fig

## 6. A 股图表分析

A 股数据用于观察 `688981.SH` 在科创板市场中的价格趋势、成交活跃度、回撤和波动。

In [ ]:
plot_single_market(a_df, 'A股 688981.SH')
plt.show()

a_summary = market_summary(a_df)
print(f"解读：A股区间累计涨跌幅为 {a_summary['累计涨跌幅%']:.2f}%，最大回撤为 {a_summary['最大回撤%']:.2f}%。")
print(f"上涨交易日占比为 {a_summary['上涨占比%']:.1f}%，说明区间内上涨和下跌天数较为接近。")

## 7. 港股图表分析

港股数据用于观察 `00981.HK` 在港交所市场中的价格趋势、成交活跃度、回撤和波动。

In [ ]:
plot_single_market(hk_df, '港股 00981.HK')
plt.show()

hk_summary = market_summary(hk_df)
print(f"解读：港股区间累计涨跌幅为 {hk_summary['累计涨跌幅%']:.2f}%，最大回撤为 {hk_summary['最大回撤%']:.2f}%。")
print(f"港股上涨交易日占比为 {hk_summary['上涨占比%']:.1f}%，成交量水平明显高于A股原始vol字段。")

## 8. A/H 股归一化收盘价对比

两地股价绝对价格和币种不同，因此直接比较价格没有意义。这里将首日收盘价统一设为 100，比较相对走势。

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(a_df['trade_date'], a_df['close_index'], label='A股 688981.SH', color=BLUE, linewidth=2)
ax.plot(hk_df['trade_date'], hk_df['close_index'], label='港股 00981.HK', color=PURPLE, linewidth=2)
ax.axhline(100, color='black', linewidth=1, alpha=0.6)
ax.set_title('图7：A股与港股归一化收盘价对比（首日=100）')
ax.set_ylabel('归一化收盘价')
ax.legend()
plt.show()

print(f"A股期末指数：{a_df['close_index'].iloc[-1]:.2f}")
print(f"港股期末指数：{hk_df['close_index'].iloc[-1]:.2f}")
print('解读：归一化指数越高，代表相对首日的累计涨幅越大；该图不比较两地绝对价格。')

## 9. 月度收益率分析

按自然月统计月初至月末收盘价变化，用于识别收益主要集中在哪些月份。

In [ ]:
def monthly_return(df: pd.DataFrame) -> pd.DataFrame:
    month = df.groupby(df['trade_date'].dt.to_period('M')).agg(
        first_close=('close', 'first'),
        last_close=('close', 'last'),
        amount=('amount', 'sum'),
        vol=('vol', 'sum'),
    )
    month['return_pct'] = (month['last_close'] / month['first_close'] - 1) * 100
    month.index = month.index.astype(str)
    return month.reset_index(names='month')

a_month = monthly_return(a_df)
hk_month = monthly_return(hk_df)

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
for ax, data, title in [(axes[0], a_month, 'A股 图8：月度收益率'), (axes[1], hk_month, '港股 图8：月度收益率')]:
    colors = np.where(data['return_pct'] >= 0, UP_COLOR, DOWN_COLOR)
    ax.bar(data['month'], data['return_pct'], color=colors)
    ax.axhline(0, color='black', linewidth=1)
    ax.set_title(title)
    ax.set_ylabel('月度收益率%')
    ax.tick_params(axis='x', rotation=45)

fig.tight_layout()
plt.show()

display(a_month[['month', 'return_pct', 'amount']])
display(hk_month[['month', 'return_pct', 'amount']])

## 10. 关键交易日定位

定位最大涨幅、最大跌幅和最高成交量交易日，辅助识别异常波动。

In [ ]:
def key_days(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    best = df.loc[df['pct_chg'].idxmax()]
    worst = df.loc[df['pct_chg'].idxmin()]
    highest_vol = df.loc[df['vol'].idxmax()]
    rows.append({'类型': '最大涨幅', '日期': best['trade_date'].date(), '涨跌幅%': best['pct_chg'], '收盘价': best['close'], '成交量': best['vol']})
    rows.append({'类型': '最大跌幅', '日期': worst['trade_date'].date(), '涨跌幅%': worst['pct_chg'], '收盘价': worst['close'], '成交量': worst['vol']})
    rows.append({'类型': '最高成交量', '日期': highest_vol['trade_date'].date(), '涨跌幅%': highest_vol['pct_chg'], '收盘价': highest_vol['close'], '成交量': highest_vol['vol']})
    return pd.DataFrame(rows)

print('A股关键交易日')
display(key_days(a_df))

print('港股关键交易日')
display(key_days(hk_df))

## 11. 结论小结

1. 本次分析将中芯国际 A 股与港股分别处理，避免混淆不同市场的交易日、价格单位和成交量口径。
2. A/H 对比使用归一化收盘价，比较的是相对收益表现，不是绝对价格水平。
3. K 线、均线、成交量、回撤、波动率和月度收益率可以从趋势、活跃度、风险和时间分布四个角度理解走势。
4. 后续如果重新通过 Tushare 更新 CSV，只需重新运行本 Notebook 即可复现全部统计图。